In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

# Objetivo

«Implementar un flujo ETL (extracción – transformación – carga) que permita tomar datos reales de disponibilidad de bicicletas en la ciudad de Londres y que almacene la información relevante en un archivo Parquet.»

# Fase de Extracción (E)

### Parámetros de conexión con la API   

In [2]:
APP_ID = os.getenv('APP_ID')
PRIMARY_KEY = os.getenv('PRIMARY_KEY')
ENDPOINT = os.getenv('ENDPOINT')

params = {
    "app_id" : APP_ID,
    "app_key": PRIMARY_KEY
}

### Respuesta del servidor

In [3]:
try:
    response = requests.get(ENDPOINT, params = params)
    response.raise_for_status()
    data = response.json()
    print(f"Se descargaron correctamente {len(data)} estaciones")

except requests.exceptions.RequestException as e:
    print(f'Error {str(e)}')

Se descargaron correctamente 799 estaciones


# Fase de Transformación (T)

In [4]:
data[0]['commonName']

'River Street , Clerkenwell'

In [5]:
data[0]['lat']

51.529163

In [6]:
data[0]['lon']

-0.10997

In [7]:
data[0]['additionalProperties']

[{'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'TerminalName',
  'sourceSystemKey': 'BikePoints',
  'value': '001023',
  'modified': '2026-05-17T23:16:11.963Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'Installed',
  'sourceSystemKey': 'BikePoints',
  'value': 'true',
  'modified': '2026-05-17T23:16:11.963Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'Locked',
  'sourceSystemKey': 'BikePoints',
  'value': 'false',
  'modified': '2026-05-17T23:16:11.963Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'InstallDate',
  'sourceSystemKey': 'BikePoints',
  'value': '1278947280000',
  'modified': '2026-05-17T23:16:11.963Z'},
 {'$type':

In [8]:
for item in data[0]['additionalProperties']:
    print(f" {item['key']} \t  {item['value']}")

 TerminalName 	  001023
 Installed 	  true
 Locked 	  false
 InstallDate 	  1278947280000
 RemovalDate 	  
 Temporary 	  false
 NbBikes 	  8
 NbEmptyDocks 	  7
 NbDocks 	  19
 NbStandardBikes 	  7
 NbEBikes 	  1


In [9]:
info_estaciones = {
    'nombre_estacion' : [], #TerminalName
    'latitud' : [], #lat
    'longitud' : [], #lon

    'disponible' : [], #Locked
    'nBicis' : [], #NnBikes
    'nEspaciosDisp' : [], #NbEmptyDocks
    'nEspacios' : [], #NbDocks
    'nBicisEstandar' : [], #NbStandardBikes
    'nBicisElectricas' : [], #NbEBikes
    'ultima_actualizacion' : []
}

### Iterar en cada estación para almacenar la información

In [10]:
for estacion in data:
    # Capa exterior
    info_estaciones['nombre_estacion'].append(estacion['commonName'])
    info_estaciones['latitud'].append(estacion['lat'])
    info_estaciones['longitud'].append(estacion['lon'])

    # Capa additionalProperties
    #Iterar por el key "additionalProperties" y extraer info actualizada de la estación
    for item in estacion['additionalProperties']:
        if item['key'] == 'NbBikes':
            info_estaciones['nBicis'].append(int(item['value']))

            # Extraer marca de tiempo última actualización
            datetime_update = pd.to_datetime(item['modified'])

            # Preservar únicamente hasta horas, minutos y segundos
            datetime_update = datetime_update.strftime("%Y-%m-%d %H:%M:%S")
            info_estaciones['ultima_actualizacion'].append(datetime_update)
            
        if item['key'] == 'NbEmptyDocks':
            info_estaciones['nEspaciosDisp'].append(int(item['value']))
        if item['key'] == 'NbDocks':
            info_estaciones['nEspacios'].append(int(item['value']))
        if item['key'] == 'NbStandardBikes':
            info_estaciones['nBicisEstandar'].append(int(item['value']))
        if item['key'] == 'NbEBikes':
            info_estaciones['nBicisElectricas'].append(int(item['value']))
        if item['key'] == 'Locked':
            if item['value']=="false":
                info_estaciones['disponible'].append('Si')
            else:
                info_estaciones['disponible'].append('No')

In [11]:
for key, value in info_estaciones.items():
    print(f"{key} : {value}")

nombre_estacion : ['River Street , Clerkenwell', 'Phillimore Gardens, Kensington', 'Christopher Street, Liverpool Street', "St. Chad's Street, King's Cross", 'Sedding Street, Sloane Square', 'Broadcasting House, Marylebone', "Charlbert Street, St. John's Wood", 'Maida Vale, Maida Vale', 'New Globe Walk, Bankside', 'Park Street, Bankside', 'Brunswick Square, Bloomsbury', 'Malet Street, Bloomsbury', 'Scala Street, Fitzrovia', 'Argyle Street, Kings Cross', 'Great Russell Street, Bloomsbury', 'Cartwright Gardens , Bloomsbury', 'Hatton Wall, Holborn', 'Drury Lane, Covent Garden', 'Taviton Street, Bloomsbury', 'Drummond Street , Euston', 'Lansdowne Drive, Hackney Central', 'Northington Street , Holborn', 'Red Lion Square, Holborn', 'British Museum, Bloomsbury', 'Doric Way , Somers Town', 'Ampton Street , Clerkenwell', 'Bolsover Street, Fitzrovia', 'Hereford Road, Bayswater', 'Windsor Terrace, Hoxton', 'Fanshaw Street, Hoxton', 'Central House, Aldgate', "Pancras Road, King's Cross", 'De Vere 

In [12]:
df = pd.DataFrame(info_estaciones)
df.head()

,nombre_estacion,latitud,longitud,disponible,nBicis,nEspaciosDisp,nEspacios,nBicisEstandar,nBicisElectricas,ultima_actualizacion
0,"River Street , Clerkenwell",51.529163,-0.109970,Si,8,7,19,7,1,2026-05-17 23:16:11
1,"Phillimore Gardens, Kensington",51.499606,-0.197574,Si,17,17,37,16,1,2026-05-17 20:46:10
2,"Christopher Street, Liverpool Street",51.521283,-0.084605,Si,0,32,32,0,0,2026-05-17 12:47:40
3,"St. Chad's Street, King's Cross",51.530059,-0.120973,Si,22,0,23,21,1,2026-05-17 21:43:39
4,"Sedding Street, Sloane Square",51.493130,-0.156876,Si,17,7,27,17,0,2026-05-17 20:39:34


# Fase de Carga (L)

In [13]:
df.to_parquet('data/estaciones.parquet')